In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/praveengovi/emotions-dataset-for-nlp/val.txt
/kaggle/input/datasets/praveengovi/emotions-dataset-for-nlp/test.txt
/kaggle/input/datasets/praveengovi/emotions-dataset-for-nlp/train.txt


In [2]:
df = pd.read_csv('/kaggle/input/datasets/praveengovi/emotions-dataset-for-nlp/train.txt',sep = ';',header = None,names = ['text','emotion'])

In [3]:
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [4]:
df.isnull().sum()

text       0
emotion    0
dtype: int64

In [5]:
unique_emonations = df['emotion'].unique()

In [6]:
emotions_number ={}
i =0
for emotion in unique_emonations:
  emotions_number[emotion] =i
  i+=1

df['emotion'] = df['emotion'].map(emotions_number)

In [7]:
df['emotion'].unique()

array([0, 1, 2, 3, 4, 5])

In [8]:
df.head()

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


In [9]:
df['text'] = df['text'].apply(lambda x:x.lower())

In [10]:
import string
def remove_punc(txt):
  return txt.translate(str.maketrans('','',string.punctuation))

In [11]:
df['text'] = df['text'].apply(remove_punc)

In [12]:
def remove_numbers(txt):
    new = ""
    for i in txt:
        if not i.isdigit():
            new = new + i
    return new

df['text'] = df['text'].apply(remove_numbers)

In [13]:
def remove_emojis(txt):
    new = ""
    for i in txt:
        if i.isascii():
            new += i
    return new

df['text'] = df['text'].apply(remove_emojis)

In [14]:
import nltk

In [15]:

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [16]:
nltk.download('punkt')
nltk.download('stopwords')


[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [17]:
stop_words = set(stopwords.words('english'))

In [18]:
df.loc[0]['text']

'i didnt feel humiliated'

In [19]:
def remove(txt):
  words = txt.split()
  cleaned = []
  for i in words:
    if not i in stop_words:
      cleaned.append(i)

  return ' '.join(cleaned)

In [20]:
df['text'] = df['text'].apply(remove)

In [21]:
df.loc[0]['text']

'didnt feel humiliated'

In [22]:
df.loc[1]['text']

'go feeling hopeless damned hopeful around someone cares awake'

In [23]:
df.head()

,text,emotion
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful around some...,0
2,im grabbing minute post feel greedy wrong,1
3,ever feeling nostalgic fireplace know still pr...,2
4,feeling grouchy,1


In [24]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df['text'], df['emotion'], test_size=0.20, random_state=42)

In [25]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)


nb_model = MultinomialNB()
nb_model.fit(X_train_bow, y_train)


pred_bow = nb_model.predict(X_test_bow)
print(accuracy_score(y_test, pred_bow))

0.768125


In [26]:
pred_bow

array([0, 5, 0, ..., 5, 5, 0])

In [55]:
tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)


nb2_model = MultinomialNB()
nb2_model.fit(X_train_tfidf,y_train)

MultinomialNB()

In [56]:
y_pred = nb2_model.predict(X_test_tfidf)

In [57]:
print(accuracy_score(y_test, y_pred))

0.701875


In [58]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(max_iter=1000)
logistic_model.fit(X_train_tfidf,y_train)


LogisticRegression(max_iter=1000)

In [59]:
log_pred = logistic_model.predict(X_test_tfidf)
print(accuracy_score(y_test,log_pred ))

0.86625


In [60]:
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

models = {
    "Logistic Regression": LogisticRegression(max_iter=2000),
    
    "Linear SVM": LinearSVC(),
    
    "SGD Classifier": SGDClassifier(
        loss="hinge",
        max_iter=2000,
        random_state=42
    ),
    
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),
    
    "KNN": KNeighborsClassifier(n_neighbors=5)
}

results = {}

for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    
    pred = model.predict(X_test_tfidf)
    accuracy = accuracy_score(y_test, pred)
    
    results[name] = accuracy
    
    print(f"{name}: {accuracy:.4f}")

Logistic Regression: 0.8662
Linear SVM: 0.9053
SGD Classifier: 0.9031
Random Forest: 0.8875
KNN: 0.7288


In [65]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

C_values = [0.5, 1, 2, 5,7,11,19]

for c in C_values:
    svm_model = LinearSVC(C=c)

    svm_model.fit(X_train_tfidf, y_train)

    y_pred = svm_model.predict(X_test_tfidf)

    accuracy = accuracy_score(y_test, y_pred)

    print(f"C = {c} → Accuracy = {accuracy:.4f}")

C = 0.5 → Accuracy = 0.9044
C = 1 → Accuracy = 0.9053
C = 2 → Accuracy = 0.9038
C = 5 → Accuracy = 0.8969
C = 7 → Accuracy = 0.8966
C = 11 → Accuracy = 0.8934
C = 19 → Accuracy = 0.8856


In [63]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

# Create SVM model
svm_model = LinearSVC(C=1)

# Train
svm_model.fit(X_train_tfidf, y_train)

# Predict
y_pred = svm_model.predict(X_test_tfidf)

# Accuracy
print("Accuracy:", accuracy_score(y_test, y_pred))

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.9053125

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.95      0.94       946
           1       0.91      0.88      0.90       427
           2       0.85      0.80      0.82       296
           3       0.86      0.72      0.78       113
           4       0.86      0.85      0.86       397
           5       0.91      0.94      0.93      1021

    accuracy                           0.91      3200
   macro avg       0.89      0.86      0.87      3200
weighted avg       0.90      0.91      0.90      3200



In [53]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
confusion_matrix(y_test, y_pred)

array([[898,  18,   6,   2,  12,  10],
       [ 20, 377,   1,   1,  15,  13],
       [  2,   2, 238,   0,   3,  51],
       [  5,   0,   1,  81,  20,   6],
       [ 20,  15,   1,   9, 339,  13],
       [ 16,   3,  34,   1,   3, 964]])

In [54]:
from sklearn.metrics import classification_report

y_pred = svm_model.predict(X_test_tfidf)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.93      0.95      0.94       946
           1       0.91      0.88      0.90       427
           2       0.85      0.80      0.82       296
           3       0.86      0.72      0.78       113
           4       0.86      0.85      0.86       397
           5       0.91      0.94      0.93      1021

    accuracy                           0.91      3200
   macro avg       0.89      0.86      0.87      3200
weighted avg       0.90      0.91      0.90      3200



In [38]:
import joblib

joblib.dump(logistic_model, "emotion_model.pkl")
joblib.dump(tfidf_vectorizer, "tfidf_vectorizer.pkl")

['tfidf_vectorizer.pkl']